# GPU runbook: OCR and Qwen3.5 on one L4

Runs on **Google Colab** with an **L4** (Runtime → Change runtime type → L4; A100 also works). **Qwen3.5 failed to start on a T4** in our attempt (vLLM 0.32, compute capability 7.5, 30 Oct 2026). This needs paid compute units; Colab's *Resources* panel shows the hourly rate.

**Set-up, as a single-GPU deployment would run it:** the vLLM server (Qwen3.5) and the OCR (PP-OCRv5 mobile through RapidOCR on ONNX Runtime CUDA) share the one GPU. vLLM reserves 80% of its memory, leaving room for OCR. Questions are processed one at a time, so OCR and the model never run at the same moment: each measured time is that of a page or call on a GPU that holds both models. Caches are off, so every call pays its full price.

**Order:**
1. Settings, Drive, packages, vLLM, code, data (cells 1–6).
2. Check on the 52 check pages (cell 7): OCR time on the shared GPU should be close to the stand-alone 0.85 s/page and model latency close to `results/check52v2-qwen3.5-4b` (e.g. DocVQA vision 1.57 s). The audit must pass.
3. OCR every page of `MANIFEST` once (cell 8, ~50 min for `full.csv`), then the model calls (cell 9; roughly 11–16 h for `full.csv`), then report and audit (cell 10).

Everything is written to **Google Drive**, so a disconnect loses nothing: run cells 1–6 again, then the cell that was interrupted; finished pages and calls are skipped. Colab sessions end after at most ~24 h, so plan on resuming once.

When you are done: **Runtime → Disconnect and delete runtime**.

In [ ]:
# 1. Settings
import os
import subprocess

MODEL = "Qwen/Qwen3.5-4B"
RUN_NAME = "full-qwen3.5-4b"  # never reuse a finished paid run's name for a different setup
MANIFEST = "benchmarks/full.csv"  # pilot: benchmarks/pilot_300.csv
CHECK = "benchmarks/check_52.csv"  # cell 7: shared-GPU check before the long run
PIPELINES = "text,vision,hybrid"
OCR = "rapid-mobile-gpu"  # PP-OCRv5 mobile on ONNX Runtime CUDA, same GPU as vLLM
GPU_MEMORY = 0.80  # vLLM's share; the rest is for OCR
DRIVE = "/content/drive/MyDrive/qdu"
OCR_CACHE = f"{DRIVE}/ocr_cache"
RUNS_DIR = f"{DRIVE}/runs"
REPO = "https://github.com/emre-geris/query-based-document-understanding.git"
# Private repo? Put a GitHub token in Colab Secrets as GITHUB_TOKEN.

SPEC = f"openai:{MODEL}"
cap = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()
print(cap)
major = float(cap.splitlines()[0].split(",")[1])
DTYPE = "bfloat16" if major >= 8 else "float16"  # T4 (7.5) has no bf16
print("dtype:", DTYPE, "| pipelines:", PIPELINES, "| OCR:", OCR)

In [ ]:
# 2. Google Drive keeps the OCR cache and the runs across sessions (a Colab disk is lost on
#    disconnect). Colab asks you to allow access the first time.
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(OCR_CACHE, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# 3. System packages: uv and vLLM (a few minutes)
!pip -q install -U uv vllm
!uv --version && python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 4. Start the vLLM server, detached (setsid) so interrupting a cell never kills it. First start
#    downloads the model (~9 GB) and compiles kernels (~7 min on L4). Re-running this cell does
#    not start a second server.
#    Prefix and multimodal caches are OFF: a cache hit on a page sent before would make the
#    later call look cheaper than it is. --max-model-len 32768: pages up to Qwen3.5's own
#    16.8 MP limit (~16k image tokens) plus OCR text. --gpu-memory-utilization leaves the rest
#    of the GPU to OCR. The flags are recorded with every run (QDU_SERVER_ARGS).
import time
import urllib.request

VLLM_FLAGS = (
    f"--port 8000 --dtype {DTYPE} --max-model-len 32768 --gpu-memory-utilization {GPU_MEMORY} "
    "--limit-mm-per-prompt '{\"image\": 1}' "
    "--no-enable-prefix-caching --mm-processor-cache-gb 0"
)
os.environ["QDU_SERVER_ARGS"] = f"vllm serve {MODEL} {VLLM_FLAGS}"


def server_ready() -> bool:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        return True
    except Exception:
        return False


running = subprocess.run(["pgrep", "-f", "[v]llm serve"], capture_output=True).returncode == 0
if not running:
    cmd = f"setsid nohup vllm serve {MODEL} {VLLM_FLAGS} > /content/vllm.log 2>&1 < /dev/null &"
    print(cmd)
    subprocess.run(cmd, shell=True)
start = time.time()
while not server_ready() and time.time() - start < 30 * 60:
    time.sleep(30)
    last = subprocess.run(["tail", "-n", "1", "/content/vllm.log"], capture_output=True, text=True)
    print(f"{int(time.time() - start):>5} s | {last.stdout.strip()[:150]}")
print("server ready" if server_ready() else "server not ready yet: check vllm.log or re-run this cell")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# 5. Code, installed with uv (uv brings its own Python 3.13; vLLM stays in the system Python).
#    The rapid-gpu extra brings RapidOCR with onnxruntime-gpu.
token = os.environ.get("GITHUB_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
os.chdir("/content")
if not os.path.isdir("qdu-repo"):
    subprocess.run(["git", "clone", "-q", url, "qdu-repo"], check=True)
os.chdir("/content/qdu-repo")
!git pull -q && git log --oneline -1
!uv sync -q --extra rapid-gpu && uv run python -c "import onnxruntime as o; print('onnxruntime', o.__version__, o.get_available_providers())"

In [ ]:
# 6. Datasets -> HF cache (~1.6 GB, all four)
!uv run qdu data download

In [ ]:
# 7. Shared-GPU check on the 52 check pages: OCR with vLLM loaded, then 3 paths, then the audit.
#    Compare with the stand-alone numbers: OCR ~0.85 s/page; model s per call as in
#    results/check52v2-qwen3.5-4b (DocVQA vision 1.57, text 0.27). Run name check52-shared-*.
!uv run qdu ocr warm --engine {OCR} --manifest {CHECK} --ocr-cache {OCR_CACHE} 2>&1 | tail -3
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --pipelines {PIPELINES} --ocr {OCR} --ocr-cache {OCR_CACHE} --manifest {CHECK} --runs-dir {RUNS_DIR} --name check52-shared-{RUN_NAME} | tail -3
!uv run qdu benchmark report {RUNS_DIR}/check52-shared-{RUN_NAME} > /dev/null && uv run qdu benchmark audit {RUNS_DIR}/check52-shared-{RUN_NAME} --examples 0

In [ ]:
# 8. OCR every page of MANIFEST once, on the shared GPU (~50 min for full.csv). Resumable.
!uv run qdu ocr warm --engine {OCR} --manifest {MANIFEST} --ocr-cache {OCR_CACHE}

In [ ]:
# 9. The run: MANIFEST x PIPELINES, one call at a time. Resumable: after a disconnect run cells
#    1-6 again, then this one.
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --pipelines {PIPELINES} --ocr {OCR} --ocr-cache {OCR_CACHE} --manifest {MANIFEST} --runs-dir {RUNS_DIR} --name {RUN_NAME}

In [ ]:
# 10. Report, audit and package (the run is also on Drive)
!uv run qdu benchmark report {RUNS_DIR}/{RUN_NAME}
!uv run qdu benchmark report {RUNS_DIR}/{RUN_NAME} --cost tokens > /dev/null
!uv run qdu benchmark audit {RUNS_DIR}/{RUN_NAME} --examples 2
!cd {RUNS_DIR} && zip -qr /content/{RUN_NAME}.zip {RUN_NAME} && ls -lh /content/{RUN_NAME}.zip